# Análisis del Abandono de Clientes (Churn) en una Entidad Bancaria Peruana

## Contexto

Una entidad bancaria con presencia nacional busca reducir la pérdida de clientes mediante el análisis de su comportamiento durante el primer semestre del año.

El banco ha observado que la cancelación de productos financieros no ocurre de manera uniforme en todas las regiones ni por todos los canales de venta. Además, la evolución mensual de la cartera de clientes evidencia diferencias importantes entre segmentos, productos y zonas geográficas.

El objetivo del proyecto es identificar los factores asociados al abandono de clientes mediante análisis descriptivos y dashboards ejecutivos desarrollados en Google Cloud, Colab, Bigquery, Looker Studio y Agentes IA, sin utilizar modelos de Machine Learning.

El análisis permitirá responder preguntas como:

- ¿Qué departamentos presentan la mayor tasa de abandono?
- ¿Qué canal de venta genera clientes con mayor permanencia?
- ¿Cómo evoluciona la cartera de clientes entre enero y junio?
- ¿Qué segmento bancario presenta mayor riesgo de abandono?
- ¿Cuál es el impacto financiero del churn por región y producto?

Además, incluiré un caso de negocio mucho más profesional

El dataset simulará la operación de un banco peruano con presencia nacional, donde los clientes pueden ser captados por distintos canales (agencias, banca digital, ejecutivos comerciales, alianzas, etc.), y el abandono estará influenciado por variables comerciales y de comportamiento. Esto permitirá construir dashboards ejecutivos como:

- Evolución mensual del churn.
- Tasa de abandono por departamento.
- Churn por canal de venta.
- Pérdida de ingresos por abandono.
- Clientes retenidos vs. perdidos.
- Análisis por segmento.
- KPIs de retención y fidelización.
- Mapa geográfico del Perú.
- Ranking de productos.
- Análisis financiero del churn.

## Diccionario de Datos


| Variable           | Tipo    | Descripción                                                                             |
| ------------------ | ------- | --------------------------------------------------------------------------------------- |
| ID_Cliente         | Long    | Identificador único del cliente                                                         |
| Fecha              | Date    | Fecha de corte mensual (enero a junio)                                                  |
| Edad               | Integer | Edad del cliente                                                                        |
| Antiguedad_Cliente | Integer | Antigüedad en meses                                                                     |
| Segmento           | String  | Premium, Preferencial, Clásico, Digital                                                 |
| Producto_Principal | String  | Cuenta de Ahorros, Cuenta Corriente, Tarjeta de Crédito, Préstamo Personal, Hipoteca    |
| Canal_Venta        | String  | Agencia, Banca Digital, Call Center, Ejecutivo Comercial, Página Web, Alianza Comercial |
| Departamento       | String  | Uno de los 24 departamentos del Perú                                                    |
| Ingreso_Mensual    | Double  | Ingreso mensual declarado                                                               |
| Saldo_Promedio     | Double  | Saldo promedio de productos                                                             |
| Numero_Productos   | Integer | Productos contratados                                                                   |
| Satisfaccion       | Integer | Escala de satisfacción (1–5)                                                            |
| Abandono           | String  | Sí / No                                                                                 |

Total: 13 variables

## Nuevas dimensiones de análisis

Con estas variables podrás construir dashboards como:

- Evolución mensual del churn.
- Churn por departamento.
- Mapa del Perú.
- Churn por canal de venta.
- Clientes nuevos vs. clientes perdidos.
- Crecimiento de cartera mensual.
- Ranking de departamentos.
- Ranking de canales.
- Participación de productos financieros.
- KPIs ejecutivos.

## Distribución de Canales de Venta

- Agencia
- Banca Digital
- Ejecutivo Comercial
- Call Center
- Página Web
- Alianza Comercial

## Departamentos del Perú

- Amazonas
- Áncash
- Apurímac
- Arequipa
- Ayacucho
- Cajamarca
- Callao
- Cusco
- Huancavelica
- Huánuco
- Ica
- Junín
- La Libertad
- Lambayeque
- Lima
- Loreto
- Madre de Dios
- Moquegua
- Pasco
- Piura
- Puno
- San Martín
- Tacna
- Tumbes

## Fechas

Una fecha aleatoria entre:

01/01/2025

y

30/06/2025

Con esto podrás comparar:

- Clientes enero
- Clientes febrero
- Clientes marzo
- Clientes abril
- Clientes mayo
- Clientes junio

y calcular la variación mensual de clientes y del churn.

# Preparación del Entorno

## Instalación del Entorno

In [ ]:
# 1. Instalar Java 17 OpenJDK (Requerido para PySpark moderno)
!apt-get install openjdk-17-jdk-headless -qq > /dev/null

# 2. Instalar la última versión de PySpark de forma silenciosa
!pip install -q pyspark

## Creación de la Sesión y Verificación

In [ ]:
import os
from pyspark.sql import SparkSession

# 3. Apuntar la variable de entorno a la nueva ruta de Java 17
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

# 4. Crear la sesión de Spark
spark = SparkSession.builder \
    .appName("MiPrimerPySparkColab") \
    .master("local[*]") \
    .getOrCreate()

# 5. Verificar que el error desapareció y ver la versión
print(f"✅ ¡Conexión exitosa con el Java Gateway!")
print(f"Versión de PySpark activa: {spark.version}")

✅ ¡Conexión exitosa con el Java Gateway!
Versión de PySpark activa: 4.0.3


# Arquitectura Medallion

## Bronze

### Creación del Dataset

In [ ]:
# ============================================================
# BLOQUE 1/4 - GOOGLE COLAB
# Instalación, SparkSession, parámetros y dim_fecha
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.storagelevel import StorageLevel

# ------------------------------------------------------------
# Parámetros generales
# ------------------------------------------------------------
SEED = 42

# 35M aprox. en la fact table: 5,833,334 clientes x 6 meses = 35,000,004 filas
TOTAL_CLIENTES = 5_833_334

# Rutas de salida en Colab
BASE_PATH = "/content/churn_banca"
RUTA_DIM_CLIENTE = f"{BASE_PATH}/dim_cliente"
RUTA_DIM_FECHA = f"{BASE_PATH}/dim_fecha"
RUTA_FACT = f"{BASE_PATH}/fact_clientes"

# ------------------------------------------------------------
# Listas maestras
# ------------------------------------------------------------
segmentos = ["Premium", "Preferencial", "Clásico", "Digital"]

canales_venta = [
    "Agencia",
    "Banca Digital",
    "Ejecutivo Comercial",
    "Call Center",
    "Página Web",
    "Alianza Comercial"
]

departamentos_peru = [
    "Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho", "Cajamarca",
    "Callao", "Cusco", "Huancavelica", "Huánuco", "Ica", "Junín",
    "La Libertad", "Lambayeque", "Lima", "Loreto", "Madre de Dios",
    "Moquegua", "Pasco", "Piura", "Puno", "San Martín", "Tacna", "Tumbes"
]

productos_principales = [
    "Cuenta de Ahorros",
    "Cuenta Corriente",
    "Tarjeta de Crédito",
    "Préstamo Personal",
    "Hipoteca"
]

# ------------------------------------------------------------
# Dimensión de fechas: enero a junio 2025
# ------------------------------------------------------------
fechas = [
    (20250101, "2025-01-01", 2025, 1, "Enero", 1),
    (20250201, "2025-02-01", 2025, 2, "Febrero", 2),
    (20250301, "2025-03-01", 2025, 3, "Marzo", 3),
    (20250401, "2025-04-01", 2025, 4, "Abril", 4),
    (20250501, "2025-05-01", 2025, 5, "Mayo", 5),
    (20250601, "2025-06-01", 2025, 6, "Junio", 6),
]

schema_fecha = T.StructType([
    T.StructField("Fecha_Key", T.IntegerType(), False),
    T.StructField("Fecha", T.StringType(), False),
    T.StructField("Anio", T.IntegerType(), False),
    T.StructField("Mes_Num", T.IntegerType(), False),
    T.StructField("Mes_Nombre", T.StringType(), False),
    T.StructField("Orden_Mes", T.IntegerType(), False),
])

dim_fecha = (
    spark.createDataFrame(fechas, schema=schema_fecha)
    .withColumn("Fecha", F.to_date("Fecha"))
)

display(dim_fecha)

DataFrame[Fecha_Key: int, Fecha: date, Anio: int, Mes_Num: int, Mes_Nombre: string, Orden_Mes: int]

In [ ]:
# ============================================================
# BLOQUE 2/4
# DimCliente: perfil del cliente y variables base
# ============================================================

df_cliente_modelo = (
    spark.range(0, TOTAL_CLIENTES, 1, numPartitions=400)
    .withColumn("ID_Cliente", (F.col("id") + F.lit(1)).cast("long"))

    # Segmento
    .withColumn("rnd_segmento", F.rand(SEED))
    .withColumn(
        "Segmento",
        F.when(F.col("rnd_segmento") < 0.10, "Premium")
         .when(F.col("rnd_segmento") < 0.32, "Preferencial")
         .when(F.col("rnd_segmento") < 0.80, "Clásico")
         .otherwise("Digital")
    )

    # Edad
    .withColumn("rnd_edad", F.rand(SEED + 1))
    .withColumn(
        "Edad",
        F.when(F.col("Segmento") == "Premium", (F.floor(F.col("rnd_edad") * 25) + 35).cast("int"))
         .when(F.col("Segmento") == "Preferencial", (F.floor(F.col("rnd_edad") * 30) + 30).cast("int"))
         .when(F.col("Segmento") == "Clásico", (F.floor(F.col("rnd_edad") * 40) + 22).cast("int"))
         .otherwise((F.floor(F.col("rnd_edad") * 28) + 18).cast("int"))
    )

    # Antigüedad inicial
    .withColumn("rnd_antig", F.rand(SEED + 2))
    .withColumn(
        "Antiguedad_Inicial",
        F.when(F.col("Segmento") == "Premium", (F.floor(F.col("rnd_antig") * 180) + 24).cast("int"))
         .when(F.col("Segmento") == "Preferencial", (F.floor(F.col("rnd_antig") * 150) + 12).cast("int"))
         .when(F.col("Segmento") == "Clásico", (F.floor(F.col("rnd_antig") * 96) + 3).cast("int"))
         .otherwise((F.floor(F.col("rnd_antig") * 72) + 1).cast("int"))
    )

    # Canal de venta
    .withColumn("rnd_canal", F.rand(SEED + 3))
    .withColumn(
        "Canal_Venta",
        F.when(F.col("Segmento") == "Premium",
               F.when(F.col("rnd_canal") < 0.35, "Ejecutivo Comercial")
                .when(F.col("rnd_canal") < 0.65, "Agencia")
                .when(F.col("rnd_canal") < 0.80, "Banca Digital")
                .when(F.col("rnd_canal") < 0.90, "Página Web")
                .when(F.col("rnd_canal") < 0.97, "Call Center")
                .otherwise("Alianza Comercial"))
         .when(F.col("Segmento") == "Preferencial",
               F.when(F.col("rnd_canal") < 0.30, "Agencia")
                .when(F.col("rnd_canal") < 0.52, "Banca Digital")
                .when(F.col("rnd_canal") < 0.70, "Ejecutivo Comercial")
                .when(F.col("rnd_canal") < 0.83, "Página Web")
                .when(F.col("rnd_canal") < 0.94, "Call Center")
                .otherwise("Alianza Comercial"))
         .when(F.col("Segmento") == "Clásico",
               F.when(F.col("rnd_canal") < 0.28, "Agencia")
                .when(F.col("rnd_canal") < 0.48, "Banca Digital")
                .when(F.col("rnd_canal") < 0.63, "Call Center")
                .when(F.col("rnd_canal") < 0.78, "Página Web")
                .when(F.col("rnd_canal") < 0.90, "Ejecutivo Comercial")
                .otherwise("Alianza Comercial"))
         .otherwise(
               F.when(F.col("rnd_canal") < 0.32, "Banca Digital")
                .when(F.col("rnd_canal") < 0.54, "Página Web")
                .when(F.col("rnd_canal") < 0.72, "Call Center")
                .when(F.col("rnd_canal") < 0.84, "Agencia")
                .when(F.col("rnd_canal") < 0.94, "Ejecutivo Comercial")
                .otherwise("Alianza Comercial"))
    )

    # Departamento
    .withColumn("rnd_depto", F.rand(SEED + 4))
    .withColumn(
        "Departamento",
        F.when(F.col("rnd_depto") < 0.34, "Lima")
         .when(F.col("rnd_depto") < 0.37, "Callao")
         .when(F.col("rnd_depto") < 0.42, "La Libertad")
         .when(F.col("rnd_depto") < 0.46, "Lambayeque")
         .when(F.col("rnd_depto") < 0.51, "Piura")
         .when(F.col("rnd_depto") < 0.56, "Arequipa")
         .when(F.col("rnd_depto") < 0.60, "Cusco")
         .when(F.col("rnd_depto") < 0.64, "Junín")
         .when(F.col("rnd_depto") < 0.68, "Áncash")
         .when(F.col("rnd_depto") < 0.71, "Ica")
         .when(F.col("rnd_depto") < 0.75, "Puno")
         .when(F.col("rnd_depto") < 0.77, "Tacna")
         .when(F.col("rnd_depto") < 0.79, "Tumbes")
         .when(F.col("rnd_depto") < 0.83, "Cajamarca")
         .when(F.col("rnd_depto") < 0.86, "San Martín")
         .when(F.col("rnd_depto") < 0.89, "Loreto")
         .when(F.col("rnd_depto") < 0.92, "Huánuco")
         .when(F.col("rnd_depto") < 0.95, "Ayacucho")
         .when(F.col("rnd_depto") < 0.97, "Apurímac")
         .when(F.col("rnd_depto") < 0.98, "Amazonas")
         .when(F.col("rnd_depto") < 0.985, "Huancavelica")
         .when(F.col("rnd_depto") < 0.99, "Moquegua")
         .when(F.col("rnd_depto") < 0.995, "Madre de Dios")
         .otherwise("Pasco")
    )

    # Producto principal
    .withColumn("rnd_producto", F.rand(SEED + 5))
    .withColumn(
        "Producto_Principal",
        F.when(F.col("Segmento") == "Premium",
               F.when(F.col("rnd_producto") < 0.30, "Cuenta Corriente")
                .when(F.col("rnd_producto") < 0.55, "Tarjeta de Crédito")
                .when(F.col("rnd_producto") < 0.75, "Hipoteca")
                .otherwise("Cuenta de Ahorros"))
         .when(F.col("Segmento") == "Preferencial",
               F.when(F.col("rnd_producto") < 0.35, "Cuenta de Ahorros")
                .when(F.col("rnd_producto") < 0.60, "Cuenta Corriente")
                .when(F.col("rnd_producto") < 0.82, "Tarjeta de Crédito")
                .otherwise("Préstamo Personal"))
         .when(F.col("Segmento") == "Clásico",
               F.when(F.col("rnd_producto") < 0.40, "Cuenta de Ahorros")
                .when(F.col("rnd_producto") < 0.65, "Tarjeta de Crédito")
                .when(F.col("rnd_producto") < 0.85, "Préstamo Personal")
                .otherwise("Cuenta Corriente"))
         .otherwise(
               F.when(F.col("rnd_producto") < 0.45, "Cuenta de Ahorros")
                .when(F.col("rnd_producto") < 0.70, "Tarjeta de Crédito")
                .when(F.col("rnd_producto") < 0.88, "Préstamo Personal")
                .otherwise("Cuenta Corriente"))
    )

    # Ingreso base
    .withColumn("rnd_ingreso", F.rand(SEED + 6))
    .withColumn(
        "Ingreso_Base",
        F.when(F.col("Segmento") == "Premium", F.round(F.col("rnd_ingreso") * 18000 + 12000, 2))
         .when(F.col("Segmento") == "Preferencial", F.round(F.col("rnd_ingreso") * 9000 + 6000, 2))
         .when(F.col("Segmento") == "Clásico", F.round(F.col("rnd_ingreso") * 4500 + 2500, 2))
         .otherwise(F.round(F.col("rnd_ingreso") * 2500 + 1000, 2))
    )

    # Saldo base
    .withColumn("rnd_saldo", F.rand(SEED + 7))
    .withColumn(
        "Saldo_Base",
        F.round(F.col("Ingreso_Base") * (F.col("rnd_saldo") * 7 + 2), 2)
    )
    .withColumn(
        "Saldo_Base",
        F.when(F.col("Producto_Principal") == "Hipoteca", F.col("Saldo_Base") * 1.80)
         .when(F.col("Producto_Principal") == "Cuenta Corriente", F.col("Saldo_Base") * 1.20)
         .when(F.col("Producto_Principal") == "Tarjeta de Crédito", F.col("Saldo_Base") * 1.08)
         .otherwise(F.col("Saldo_Base"))
    )

    # Número de productos base
    .withColumn("rnd_prod", F.rand(SEED + 8))
    .withColumn(
        "Numero_Productos_Base",
        F.when(F.col("Segmento") == "Premium", (F.floor(F.col("rnd_prod") * 3) + 4).cast("int"))
         .when(F.col("Segmento") == "Preferencial", (F.floor(F.col("rnd_prod") * 3) + 3).cast("int"))
         .when(F.col("Segmento") == "Clásico", (F.floor(F.col("rnd_prod") * 3) + 2).cast("int"))
         .otherwise((F.floor(F.col("rnd_prod") * 3) + 1).cast("int"))
    )

    # Satisfacción base
    .withColumn("rnd_satisf", F.rand(SEED + 9))
    .withColumn(
        "Satisfaccion_Base",
        F.when(F.col("Canal_Venta") == "Ejecutivo Comercial", (F.floor(F.col("rnd_satisf") * 2) + 4).cast("int"))
         .when(F.col("Canal_Venta") == "Agencia", (F.floor(F.col("rnd_satisf") * 3) + 3).cast("int"))
         .when(F.col("Canal_Venta") == "Banca Digital", (F.floor(F.col("rnd_satisf") * 4) + 2).cast("int"))
         .when(F.col("Canal_Venta") == "Página Web", (F.floor(F.col("rnd_satisf") * 4) + 2).cast("int"))
         .when(F.col("Canal_Venta") == "Call Center", (F.floor(F.col("rnd_satisf") * 3) + 1).cast("int"))
         .otherwise((F.floor(F.col("rnd_satisf") * 5) + 1).cast("int"))
    )
    .withColumn(
        "Satisfaccion_Base",
        F.when(
            (F.col("Segmento") == "Premium") & (F.col("Satisfaccion_Base") < 5),
            F.col("Satisfaccion_Base") + 1
        )
        .when(
            (F.col("Segmento") == "Digital") & (F.col("Satisfaccion_Base") > 1),
            F.col("Satisfaccion_Base") - 1
        )
        .otherwise(F.col("Satisfaccion_Base"))
    )
    .withColumn(
        "Satisfaccion_Base",
        F.least(F.lit(5), F.greatest(F.lit(1), F.col("Satisfaccion_Base")))
    )

    # Score de riesgo base
    .withColumn(
        "Score_Riesgo_Base",
        F.lit(0)
        + F.when(F.col("Satisfaccion_Base") == 1, 40).when(F.col("Satisfaccion_Base") == 2, 30).when(F.col("Satisfaccion_Base") == 3, 15).otherwise(0)
        + F.when(F.col("Antiguedad_Inicial") < 12, 25).when(F.col("Antiguedad_Inicial") < 24, 15).when(F.col("Antiguedad_Inicial") < 48, 5).otherwise(0)
        + F.when(F.col("Numero_Productos_Base") == 1, 20).when(F.col("Numero_Productos_Base") == 2, 10).otherwise(0)
        + F.when(F.col("Segmento") == "Digital", 15).when(F.col("Segmento") == "Clásico", 8).when(F.col("Segmento") == "Preferencial", 3).otherwise(0)
        + F.when(F.col("Canal_Venta") == "Call Center", 15).when(F.col("Canal_Venta") == "Página Web", 10).when(F.col("Canal_Venta") == "Banca Digital", 8).when(F.col("Canal_Venta") == "Alianza Comercial", 6).otherwise(0)
        + F.when(F.col("Departamento").isin("Loreto", "Madre de Dios", "Amazonas"), 8)
           .when(F.col("Departamento").isin("Huancavelica", "Pasco", "Apurímac"), 6)
           .otherwise(0)
        + F.when(F.col("Ingreso_Base") < 2500, 10).when(F.col("Ingreso_Base") < 4000, 5).otherwise(0)
        + F.when(F.col("Saldo_Base") < 3000, 10).when(F.col("Saldo_Base") < 10000, 5).otherwise(0)
    )

    # Probabilidad de churn a 6 meses
    .withColumn(
        "Prob_Churn_6m",
        F.least(
            F.lit(0.38),
            F.greatest(
                F.lit(0.05),
                F.lit(0.05) + (F.col("Score_Riesgo_Base") / F.lit(220.0))
            )
        )
    )

    # Churn dentro del periodo
    .withColumn("rnd_churn", F.rand(SEED + 10))
    .withColumn(
        "Flag_Churn_6m",
        F.when(F.col("rnd_churn") < F.col("Prob_Churn_6m"), F.lit(1)).otherwise(F.lit(0))
    )

    # Mes de baja
    .withColumn("rnd_mes_baja", F.rand(SEED + 11))
    .withColumn(
        "Mes_Baja",
        F.when(
            F.col("Flag_Churn_6m") == 1,
            F.when(F.col("rnd_mes_baja") < 0.14, 1)
             .when(F.col("rnd_mes_baja") < 0.30, 2)
             .when(F.col("rnd_mes_baja") < 0.47, 3)
             .when(F.col("rnd_mes_baja") < 0.66, 4)
             .when(F.col("rnd_mes_baja") < 0.83, 5)
             .otherwise(6)
        ).otherwise(F.lit(7))
    )
)

# Dimensión final del cliente: solo atributos estables
dim_cliente = (
    df_cliente_modelo
    .select(
        "ID_Cliente",
        "Edad",
        "Antiguedad_Inicial",
        "Segmento",
        "Canal_Venta",
        "Departamento",
        "Producto_Principal"
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)

# Tabla auxiliar completa para construir la fact
df_cliente_modelo = (
    df_cliente_modelo
    .select(
        "ID_Cliente",
        "Edad",
        "Antiguedad_Inicial",
        "Segmento",
        "Canal_Venta",
        "Departamento",
        "Producto_Principal",
        "Ingreso_Base",
        "Saldo_Base",
        "Numero_Productos_Base",
        "Satisfaccion_Base",
        "Mes_Baja"
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)

display(dim_cliente.limit(10))

DataFrame[ID_Cliente: bigint, Edad: int, Antiguedad_Inicial: int, Segmento: string, Canal_Venta: string, Departamento: string, Producto_Principal: string]

In [ ]:
# ============================================================
# BLOQUE 3/4
# FactClientes: snapshot mensual enero-junio
# ============================================================

fact_clientes = (
    df_cliente_modelo
    .crossJoin(F.broadcast(dim_fecha))
    .withColumn(
        "Antiguedad_Meses",
        (F.col("Antiguedad_Inicial") + F.col("Orden_Mes") - F.lit(1)).cast("int")
    )

    # Estado del cliente por mes
    .withColumn(
        "Es_Abandono",
        F.when(F.col("Orden_Mes") == F.col("Mes_Baja"), F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "Es_Activo",
        F.when(F.col("Orden_Mes") < F.col("Mes_Baja"), F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn(
        "Estado_Cliente",
        F.when(F.col("Es_Activo") == 1, F.lit("Activo")).otherwise(F.lit("Baja"))
    )
    .withColumn(
        "Abandono",
        F.when(F.col("Es_Abandono") == 1, F.lit("Sí")).otherwise(F.lit("No"))
    )

    # Ingreso mensual
    .withColumn(
        "Ingreso_Mensual",
        F.when(
            F.col("Es_Activo") == 1,
            F.round(
                F.col("Ingreso_Base")
                * (1 + (F.col("Orden_Mes") - 1) * 0.006)
                * (1 + (F.rand(SEED + 20) - 0.5) * 0.06),
                2
            )
        ).otherwise(F.lit(None).cast("double"))
    )

    # Saldo promedio
    .withColumn(
        "Saldo_Promedio",
        F.when(
            F.col("Es_Activo") == 1,
            F.round(
                F.col("Saldo_Base")
                * (1 + (F.col("Orden_Mes") - 1) * 0.010)
                * (1 + (F.rand(SEED + 21) - 0.5) * 0.08),
                2
            )
        ).otherwise(F.lit(None).cast("double"))
    )

    # Número de productos
    .withColumn(
        "Numero_Productos",
        F.when(
            F.col("Es_Activo") == 1,
            F.when(
                (F.col("Orden_Mes") >= 4) & (F.rand(SEED + 22) < 0.05) & (F.col("Numero_Productos_Base") < 6),
                F.col("Numero_Productos_Base") + 1
            ).otherwise(F.col("Numero_Productos_Base"))
        ).otherwise(F.lit(None).cast("int"))
    )

    # Satisfacción mensual
    .withColumn(
        "Satisfaccion",
        F.when(
            F.col("Es_Activo") == 1,
            F.least(
                F.lit(5),
                F.greatest(
                    F.lit(1),
                    F.round(
                        F.col("Satisfaccion_Base") + ((F.rand(SEED + 23) - 0.5) * 2),
                        0
                    ).cast("int")
                )
            )
        ).otherwise(F.lit(None).cast("int"))
    )
)

fact_clientes = fact_clientes.select(
    "ID_Cliente",
    "Fecha_Key",
    "Antiguedad_Meses",
    "Ingreso_Mensual",
    "Saldo_Promedio",
    "Numero_Productos",
    "Satisfaccion",
    "Es_Activo",
    "Es_Abandono",
    "Abandono",
    "Estado_Cliente"
)

display(fact_clientes.limit(20))

DataFrame[ID_Cliente: bigint, Fecha_Key: int, Antiguedad_Meses: int, Ingreso_Mensual: double, Saldo_Promedio: double, Numero_Productos: int, Satisfaccion: int, Es_Activo: int, Es_Abandono: int, Abandono: string, Estado_Cliente: string]

In [ ]:
# ============================================================
# BLOQUE 4/4
# Validación y escritura en Parquet sin particionar
# ============================================================

# ------------------------------------------------------------
# Repartición recomendada para escritura
# ------------------------------------------------------------
dim_cliente_write = dim_cliente.repartition(200)
dim_fecha_write = dim_fecha.repartition(1)
fact_write = fact_clientes.repartition(240)

# ------------------------------------------------------------
# Validaciones
# ------------------------------------------------------------
print("Conteo DimFecha:", dim_fecha_write.count())
print("Conteo DimCliente:", dim_cliente_write.count())
print("Conteo FactClientes:", fact_write.count())

print("\nDistribución del churn en la fact table:")
fact_write.groupBy("Abandono").count().show()

print("\nClientes activos por mes:")
(
    fact_write
    .groupBy("Fecha_Key")
    .agg(
        F.sum("Es_Activo").alias("Clientes_Activos"),
        F.sum("Es_Abandono").alias("Clientes_Abandonados")
    )
    .orderBy("Fecha_Key")
    .show()
)

print("\nAbandono por segmento:")
(
    df_cliente_modelo
    .join(fact_write.select("ID_Cliente", "Fecha_Key", "Es_Abandono"), "ID_Cliente", "inner")
    .groupBy("Segmento")
    .agg(F.avg("Es_Abandono").alias("Tasa_Abandono_Aprox"))
    .show()
)

# ------------------------------------------------------------
# Escritura en Parquet sin particionar
# ------------------------------------------------------------
(
    dim_fecha_write
    .write
    .mode("overwrite")
    .option("compression", "snappy")
    .parquet(RUTA_DIM_FECHA)
)

(
    dim_cliente_write
    .write
    .mode("overwrite")
    .option("compression", "snappy")
    .parquet(RUTA_DIM_CLIENTE)
)

(
    fact_write
    .write
    .mode("overwrite")
    .option("compression", "snappy")
    .parquet(RUTA_FACT)
)

print("\nArchivos generados correctamente:")
print("DimFecha ->", RUTA_DIM_FECHA)
print("DimCliente ->", RUTA_DIM_CLIENTE)
print("FactClientes ->", RUTA_FACT)

# ------------------------------------------------------------
# Vistas temporales para análisis
# ------------------------------------------------------------
dim_fecha.createOrReplaceTempView("vw_dim_fecha")
dim_cliente.createOrReplaceTempView("vw_dim_cliente")
fact_clientes.createOrReplaceTempView("vw_fact_clientes")

# ------------------------------------------------------------
# Limpieza
# ------------------------------------------------------------
df_cliente_modelo.unpersist()
dim_cliente.unpersist()

display(fact_clientes.limit(20))

Conteo DimFecha: 6
Conteo DimCliente: 5833334
Conteo FactClientes: 35000004

Distribución del churn en la fact table:
+--------+--------+
|Abandono|   count|
+--------+--------+
|      No|33772660|
|      Sí| 1227344|
+--------+--------+


Clientes activos por mes:
+---------+----------------+--------------------+
|Fecha_Key|Clientes_Activos|Clientes_Abandonados|
+---------+----------------+--------------------+
| 20250101|         5661000|              172334|
| 20250201|         5464677|              196323|
| 20250301|         5256207|              208470|
| 20250401|         5023428|              232779|
| 20250501|         4814778|              208650|
| 20250601|         4605990|              208788|
+---------+----------------+--------------------+


Abandono por segmento:
+------------+--------------------+
|    Segmento| Tasa_Abandono_Aprox|
+------------+--------------------+
|     Premium|0.013486343398253954|
|     Clásico| 0.03577327366236243|
|Preferencial|0.0235880889524

DataFrame[ID_Cliente: bigint, Fecha_Key: int, Antiguedad_Meses: int, Ingreso_Mensual: double, Saldo_Promedio: double, Numero_Productos: int, Satisfaccion: int, Es_Activo: int, Es_Abandono: int, Abandono: string, Estado_Cliente: string]

## Silver

### Limpieza y Verificación de Nulos

In [ ]:
from pyspark.sql import functions as F

def verificar_nulos_y_vacios(view_name):
    print(f"\n--- Verificando nulos y vacíos en: {view_name} ---")
    df = spark.table(view_name)

    # Crear expresiones de conteo para cada columna
    # Cuenta si es nulo O si es una cadena vacía (después de quitar espacios)
    expresiones = [
        F.count(F.when(F.col(c).isNull() | (F.col(c).cast("string") == ""), c)).alias(c)
        for c in df.columns
    ]

    # Ejecutar la agregación
    resultado = df.select(expresiones).collect()[0].asDict()

    # Mostrar resultados solo de columnas que tengan al menos un problema
    hay_problemas = False
    for col, count in resultado.items():
        if count > 0:
            print(f"❌ Columna '{col}': {count} valores nulos/vacíos detected.")
            hay_problemas = True

    if not hay_problemas:
        print(f"✅ ¡Excelente! No se encontraron nulos ni vacíos en {view_name}.")

# Ejecutar la verificación para cada vista
vistas = ["vw_dim_fecha", "vw_dim_cliente", "vw_fact_clientes"]
for vista in vistas:
    verificar_nulos_y_vacios(vista)


--- Verificando nulos y vacíos en: vw_dim_fecha ---
✅ ¡Excelente! No se encontraron nulos ni vacíos en vw_dim_fecha.

--- Verificando nulos y vacíos en: vw_dim_cliente ---
✅ ¡Excelente! No se encontraron nulos ni vacíos en vw_dim_cliente.

--- Verificando nulos y vacíos en: vw_fact_clientes ---
❌ Columna 'Ingreso_Mensual': 4173924 valores nulos/vacíos detected.
❌ Columna 'Saldo_Promedio': 4173924 valores nulos/vacíos detected.
❌ Columna 'Numero_Productos': 4173924 valores nulos/vacíos detected.
❌ Columna 'Satisfaccion': 4173924 valores nulos/vacíos detected.


In [ ]:
from pyspark.sql import functions as F

# Obtener la tabla de la vista
df_fact = spark.table("vw_fact_clientes")

# Crear una condición dinámica para detectar nulos o vacíos en cualquier columna
condicion = None
for col_name in df_fact.columns:
    col_cond = F.col(col_name).isNull() | (F.col(col_name).cast("string") == "")
    if condicion is None:
        condicion = col_cond
    else:
        condicion = condicion | col_cond

# Filtrar los registros con problemas y limitar a 30
df_nulos_vacios = df_fact.filter(condicion).limit(30)

# Mostrar metadatos
print("--- Esquema de la tabla (Metadatos) ---")
df_nulos_vacios.printSchema()

# Mostrar data forzando la salida en consola y luego en display
print("\n--- Primeros 30 registros con nulos o vacíos (Vista de tabla) ---")
df_nulos_vacios.show(30, truncate=False)

print("\n--- Vista Interactiva ---")
display(df_nulos_vacios)

--- Esquema de la tabla (Metadatos) ---
root
 |-- ID_Cliente: long (nullable = false)
 |-- Fecha_Key: integer (nullable = false)
 |-- Antiguedad_Meses: integer (nullable = true)
 |-- Ingreso_Mensual: double (nullable = true)
 |-- Saldo_Promedio: double (nullable = true)
 |-- Numero_Productos: integer (nullable = true)
 |-- Satisfaccion: integer (nullable = true)
 |-- Es_Activo: integer (nullable = false)
 |-- Es_Abandono: integer (nullable = false)
 |-- Abandono: string (nullable = false)
 |-- Estado_Cliente: string (nullable = false)


--- Primeros 30 registros con nulos o vacíos (Vista de tabla) ---
+----------+---------+----------------+---------------+--------------+----------------+------------+---------+-----------+--------+--------------+
|ID_Cliente|Fecha_Key|Antiguedad_Meses|Ingreso_Mensual|Saldo_Promedio|Numero_Productos|Satisfaccion|Es_Activo|Es_Abandono|Abandono|Estado_Cliente|
+----------+---------+----------------+---------------+--------------+----------------+----------

DataFrame[ID_Cliente: bigint, Fecha_Key: int, Antiguedad_Meses: int, Ingreso_Mensual: double, Saldo_Promedio: double, Numero_Productos: int, Satisfaccion: int, Es_Activo: int, Es_Abandono: int, Abandono: string, Estado_Cliente: string]

Verificamos si efectivamente los clientes con registros Nulos corresponden a clientes con Estado_Cliente igual a "Baja" con motivo de constatar si es intencional (porque el cliente ya no está activo).

In [ ]:
import pyspark.sql.functions as F

# 1. Identificar registros con nulos (usando la lógica de la celda anterior)
# Reutilizamos df_fact que ya existe en el kernel
df_fact = spark.table("vw_fact_clientes")

condicion_nulos = None
for col_name in ["Ingreso_Mensual", "Saldo_Promedio", "Numero_Productos", "Satisfaccion"]:
    col_cond = F.col(col_name).isNull()
    if condicion_nulos is None:
        condicion_nulos = col_cond
    else:
        condicion_nulos = condicion_nulos | col_cond

# 2. Contar cuántos registros con nulos hay
total_nulos = df_fact.filter(condicion_nulos).count()

# 3. Contar cuántos registros con nulos TIENEN Estado_Cliente == 'Baja'
nulos_estado_baja = df_fact.filter(condicion_nulos & (F.col("Estado_Cliente") == "Baja")).count()

# 4. Mostrar resultados
print(f"Total de registros con valores nulos: {total_nulos}")
print(f"Total de registros con nulos que están en 'Baja': {nulos_estado_baja}")

if total_nulos == nulos_estado_baja:
    print("\n✅ Confirmado: El 100% de los registros con nulos pertenecen a clientes en estado 'Baja'.")
else:
    print(f"\n⚠️ Atención: Hay {total_nulos - nulos_estado_baja} registros con nulos que NO están en 'Baja'.")

# Visualización rápida de la relación
df_fact.filter(condicion_nulos).groupBy("Estado_Cliente").count().show()

Total de registros con valores nulos: 4173924
Total de registros con nulos que están en 'Baja': 4173924

✅ Confirmado: El 100% de los registros con nulos pertenecen a clientes en estado 'Baja'.
+--------------+-------+
|Estado_Cliente|  count|
+--------------+-------+
|          Baja|4173924|
+--------------+-------+



Reemplazamos todos los valores nulos por 0 del dataframe "df_fact". El resultado anterior confirma que los datos nulos corresponden a clientes que ya no están activos.

In [ ]:
# Reemplazar todos los valores nulos por 0 en el DataFrame df_fact
df_fact_limpio = df_fact.fillna(0)

# Actualizar la vista temporal con los datos limpios
df_fact_limpio.createOrReplaceTempView("vw_fact_clientes")

# Verificar que ya no existen nulos en las columnas críticas
print("Verificación tras reemplazo:")
df_fact_limpio.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ["Ingreso_Mensual", "Saldo_Promedio", "Numero_Productos", "Satisfaccion"]
]).show()

# Mostrar una muestra de los datos actualizados
display(df_fact_limpio.limit(10))

Verificación tras reemplazo:
+---------------+--------------+----------------+------------+
|Ingreso_Mensual|Saldo_Promedio|Numero_Productos|Satisfaccion|
+---------------+--------------+----------------+------------+
|              0|             0|               0|           0|
+---------------+--------------+----------------+------------+



DataFrame[ID_Cliente: bigint, Fecha_Key: int, Antiguedad_Meses: int, Ingreso_Mensual: double, Saldo_Promedio: double, Numero_Productos: int, Satisfaccion: int, Es_Activo: int, Es_Abandono: int, Abandono: string, Estado_Cliente: string]

### Persistencia en Capa Silver
En este paso, guardamos los DataFrames procesados como tablas persistentes en el catálogo de Spark para facilitar su consumo en la capa Gold o herramientas de BI.

In [ ]:
import os

# --- BLOQUE 1: Persistencia Física en Parquet ---
PATH_SILVER = "/content/churn_banca/silver"
os.makedirs(PATH_SILVER, exist_ok=True)

print(f"Guardando archivos Parquet en: {PATH_SILVER}...")

# Guardar físicamente los DataFrames resultantes
df_fact_limpio.write.mode("overwrite").parquet(f"{PATH_SILVER}/fact_silver")
dim_fecha.write.mode("overwrite").parquet(f"{PATH_SILVER}/dim_fecha")
dim_cliente.write.mode("overwrite").parquet(f"{PATH_SILVER}/dim_clientes")

print("✅ Archivos guardados exitosamente en disco.")

Guardando archivos Parquet en: /content/churn_banca/silver...
✅ Archivos guardados exitosamente en disco.


In [ ]:
# --- BLOQUE 2: Registro en el Catálogo de Spark ---
PATH_SILVER = "/content/churn_banca/silver"

print("Registrando tablas en el catálogo desde los archivos Parquet...")

# Leer los archivos guardados y crear vistas/tablas temporales
spark.read.parquet(f"{PATH_SILVER}/fact_silver").createOrReplaceTempView("df_fact_silver")
spark.read.parquet(f"{PATH_SILVER}/dim_fecha").createOrReplaceTempView("dim_fecha")
spark.read.parquet(f"{PATH_SILVER}/dim_clientes").createOrReplaceTempView("dim_clientes")

# Verificar registro
print("✅ Tablas disponibles en la sesión actual:")
spark.sql("SHOW TABLES").show()

Registrando tablas en el catálogo desde los archivos Parquet...
✅ Tablas disponibles en la sesión actual:
+---------+----------------+-----------+
|namespace|       tableName|isTemporary|
+---------+----------------+-----------+
|         |  df_fact_silver|       true|
|         |    dim_clientes|       true|
|         |       dim_fecha|       true|
|         |  vw_dim_cliente|       true|
|         |    vw_dim_fecha|       true|
|         |vw_fact_clientes|       true|
+---------+----------------+-----------+



## Gold

Analizamos que factores estan asociados con el abandono de clienes, que factores son los responsables del retiro de los clientes.

### 1º Calculamos el indicador principal: la tasa de Churn

In [ ]:
from pyspark.sql import functions as F

# Cargar la tabla de hechos desde la capa silver
df_gold_churn = spark.table("df_fact_silver")

# Calcular KPI global de Churn
kpi_churn = df_gold_churn.select(
    (F.sum("Es_Abandono") / F.countDistinct("ID_Cliente") * 100).alias("Tasa_Churn_Global")
)

print("--- Indicador Principal (Gold) ---")
kpi_churn.show()

# También podemos ver la evolución mensual de la tasa de churn
churn_mensual = df_gold_churn.groupBy("Fecha_Key").agg(
    F.sum("Es_Activo").alias("Clientes_Activos"),
    F.sum("Es_Abandono").alias("Clientes_Baja"),
    (F.sum("Es_Abandono") / F.sum("Es_Activo") * 100).alias("Tasa_Churn_Mensual")
).orderBy("Fecha_Key")

print("--- Evolución Mensual del Churn ---")
churn_mensual.show()

--- Indicador Principal (Gold) ---
+------------------+
| Tasa_Churn_Global|
+------------------+
|21.040180452550807|
+------------------+

--- Evolución Mensual del Churn ---
+---------+----------------+-------------+------------------+
|Fecha_Key|Clientes_Activos|Clientes_Baja|Tasa_Churn_Mensual|
+---------+----------------+-------------+------------------+
| 20250101|         5661000|       172334|3.0442324677618795|
| 20250201|         5464677|       196323|3.5925819586409222|
| 20250301|         5256207|       208470|  3.96616799909136|
| 20250401|         5023428|       232779| 4.633867550206752|
| 20250501|         4814778|       208650| 4.333533134861047|
| 20250601|         4605990|       208788| 4.532966854031381|
+---------+----------------+-------------+------------------+



### 2° Calculamos la relación entre el abandono y las demás variables

In [ ]:
from pyspark.sql import functions as F

# 1. Preparar el dataset uniendo Hechos con Dimensiones
df_analisis = spark.table("df_fact_silver").join(
    spark.table("dim_clientes"),
    "ID_Cliente",
    "inner"
)

def analizar_churn_por_columna(df, columna):
    print(f"\n--- Análisis de Churn por {columna} ---")
    resultado = df.groupBy(columna).agg(
        F.countDistinct("ID_Cliente").alias("Total_Clientes"),
        F.sum("Es_Abandono").alias("Total_Bajas"),
        (F.sum("Es_Abandono") / F.countDistinct("ID_Cliente") * 100).alias("Tasa_Churn_%")
    ).orderBy(F.desc("Tasa_Churn_%"))
    resultado.show()

# 2. Ejecutar análisis para variables clave
variables_interes = ["Segmento", "Canal_Venta", "Producto_Principal", "Departamento"]

for var in variables_interes:
    analizar_churn_por_columna(df_analisis, var)


--- Análisis de Churn por Segmento ---
+------------+--------------+-----------+------------------+
|    Segmento|Total_Clientes|Total_Bajas|      Tasa_Churn_%|
+------------+--------------+-----------+------------------+
|     Digital|       1167414|     397855| 34.08002645162727|
|     Clásico|       2797787|     600516| 21.46396419741746|
|Preferencial|       1283727|     181684|14.152853371472283|
|     Premium|        584406|      47289| 8.091806038952372|
+------------+--------------+-----------+------------------+


--- Análisis de Churn por Canal_Venta ---
+-------------------+--------------+-----------+------------------+
|        Canal_Venta|Total_Clientes|Total_Bajas|      Tasa_Churn_%|
+-------------------+--------------+-----------+------------------+
|        Call Center|        813332|     262785|32.309684114236255|
|         Página Web|        901089|     223075|24.756156162154905|
|  Alianza Comercial|        444286|     107641|24.227862232886025|
|      Banca Digital

In [ ]:
from pyspark.sql import functions as F

# Deshabilitar el broadcast join automático para evitar el error de memoria (Py4JJavaError)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

# Unimos hechos y dimensiones para tener el panorama completo
df_full = spark.table("df_fact_silver").join(
    spark.table("dim_clientes"),
    "ID_Cliente",
    "inner"
)

# 1. Análisis por Nivel de Satisfacción (Variable crítica)
print("--- Tasa de Abandono por Nivel de Satisfacción ---")
df_full.filter("Estado_Cliente == 'Activo' OR Es_Abandono == 1").groupBy("Satisfaccion").agg(
    F.countDistinct("ID_Cliente").alias("Total_Clientes"),
    F.sum("Es_Abandono").alias("Bajas"),
    (F.sum("Es_Abandono") / F.countDistinct("ID_Cliente") * 100).alias("Tasa_Churn_%")
).orderBy("Satisfaccion").show()

# 2. Análisis por Número de Productos (Cross-selling)
print("--- Tasa de Abandono por Número de Productos ---")
df_full.filter("Estado_Cliente == 'Activo' OR Es_Abandono == 1").groupBy("Numero_Productos").agg(
    F.countDistinct("ID_Cliente").alias("Total_Clientes"),
    F.sum("Es_Abandono").alias("Bajas"),
    (F.sum("Es_Abandono") / F.countDistinct("ID_Cliente") * 100).alias("Tasa_Churn_%")
).orderBy("Numero_Productos").show()

# 3. Análisis por Rangos de Edad
print("--- Tasa de Abandono por Rango de Edad ---")
df_full.withColumn(
    "Rango_Edad",
    F.when(F.col("Edad") < 30, "18-29")
     .when(F.col("Edad") < 45, "30-44")
     .when(F.col("Edad") < 60, "45-59")
     .otherwise("60+")
).groupBy("Rango_Edad").agg(
    F.countDistinct("ID_Cliente").alias("Total_Clientes"),
    F.sum("Es_Abandono").alias("Bajas"),
    (F.sum("Es_Abandono") / F.countDistinct("ID_Cliente") * 100).alias("Tasa_Churn_%")
).orderBy("Rango_Edad").show()

# 4. Análisis por Nivel de Ingresos
print("--- Tasa de Abandono por Rango de Ingresos ---")
df_full.withColumn(
    "Rango_Ingreso",
    F.when(F.col("Ingreso_Mensual") < 2500, "Bajo (<2.5k)")
     .when(F.col("Ingreso_Mensual") < 6000, "Medio (2.5k - 6k)")
     .when(F.col("Ingreso_Mensual") < 12000, "Alto (6k - 12k)")
     .otherwise("Top (>12k)")
).groupBy("Rango_Ingreso").agg(
    F.countDistinct("ID_Cliente").alias("Total_Clientes"),
    F.sum("Es_Abandono").alias("Bajas"),
    (F.sum("Es_Abandono") / F.countDistinct("ID_Cliente") * 100).alias("Tasa_Churn_%")
).orderBy(F.desc("Tasa_Churn_%")).show()

# 5. Análisis por Saldo Promedio
print("--- Tasa de Abandono por Rango de Saldo Promedio ---")
df_full.withColumn(
    "Rango_Saldo",
    F.when(F.col("Saldo_Promedio") < 5000, "Bajo (<5k)")
     .when(F.col("Saldo_Promedio") < 20000, "Medio (5k - 20k)")
     .when(F.col("Saldo_Promedio") < 50000, "Alto (20k - 50k)")
     .otherwise("Top (>50k)")
).groupBy("Rango_Saldo").agg(
    F.countDistinct("ID_Cliente").alias("Total_Clientes"),
    F.sum("Es_Abandono").alias("Bajas"),
    (F.sum("Es_Abandono") / F.countDistinct("ID_Cliente") * 100).alias("Tasa_Churn_%")
).orderBy(F.desc("Tasa_Churn_%")).show()

--- Tasa de Abandono por Nivel de Satisfacción ---
+------------+--------------+-------+------------+
|Satisfaccion|Total_Clientes|  Bajas|Tasa_Churn_%|
+------------+--------------+-------+------------+
|           0|       1227344|1227344|       100.0|
|           1|       1196408|      0|         0.0|
|           2|       2212745|      0|         0.0|
|           3|       3019046|      0|         0.0|
|           4|       3559407|      0|         0.0|
|           5|       2613259|      0|         0.0|
+------------+--------------+-------+------------+

--- Tasa de Abandono por Número de Productos ---
+----------------+--------------+-------+------------+
|Numero_Productos|Total_Clientes|  Bajas|Tasa_Churn_%|
+----------------+--------------+-------+------------+
|               0|       1227344|1227344|       100.0|
|               1|        369430|      0|         0.0|
|               2|       1310259|      0|         0.0|
|               3|       1843900|      0|         0.0|
|   

In [ ]:
from pyspark.sql import functions as F

# Usamos la tabla de hechos de la capa Silver
df_antiguedad = spark.table("df_fact_silver")

# 1. Crear rangos de antigüedad para un análisis más claro
print("--- Tasa de Abandono por Rango de Antigüedad (Meses) ---")
df_antiguedad_analisis = df_antiguedad.withColumn(
    "Rango_Antiguedad",
    F.when(F.col("Antiguedad_Meses") < 12, "0-1 año")
     .when(F.col("Antiguedad_Meses") < 24, "1-2 años")
     .when(F.col("Antiguedad_Meses") < 48, "2-4 años")
     .when(F.col("Antiguedad_Meses") < 84, "4-7 años")
     .otherwise("7+ años")
)

# 2. Calcular la tasa de churn por rango
df_antiguedad_analisis.groupBy("Rango_Antiguedad").agg(
    F.countDistinct("ID_Cliente").alias("Total_Clientes"),
    F.sum("Es_Abandono").alias("Bajas"),
    (F.sum("Es_Abandono") / F.countDistinct("ID_Cliente") * 100).alias("Tasa_Churn_%")
).orderBy(F.desc("Tasa_Churn_%")).show()

# 3. Análisis detallado: ¿Hay un mes específico con picos de abandono?
print("\n--- Top 10 Meses Específicos de Antigüedad con Mayor Churn ---")
df_antiguedad.groupBy("Antiguedad_Meses").agg(
    F.countDistinct("ID_Cliente").alias("Total_Clientes"),
    F.sum("Es_Abandono").alias("Bajas"),
    (F.sum("Es_Abandono") / F.countDistinct("ID_Cliente") * 100).alias("Tasa_Churn_%")
).orderBy(F.desc("Tasa_Churn_%")).limit(10).show()

--- Tasa de Abandono por Rango de Antigüedad (Meses) ---
+----------------+--------------+------+------------------+
|Rango_Antiguedad|Total_Clientes| Bajas|      Tasa_Churn_%|
+----------------+--------------+------+------------------+
|         0-1 año|        441548|105537|  23.9015916729325|
|        1-2 años|        875967|179161| 20.45293943721624|
|        2-4 años|       1640281|323800|19.740520069427127|
|        4-7 años|       2163167|398351|18.415175527363353|
|         7+ años|       1700407|220495|12.967189619896882|
+----------------+--------------+------+------------------+


--- Top 10 Meses Específicos de Antigüedad con Mayor Churn ---
+----------------+--------------+-----+------------------+
|Antiguedad_Meses|Total_Clientes|Bajas|      Tasa_Churn_%|
+----------------+--------------+-----+------------------+
|               2|         32504| 1793| 5.516244154565592|
|               6|        213412|11725|5.4940678124941424|
|               7|        243016|13121|5.39

### 3° Analizamos la relación bivariable

In [ ]:
from pyspark.sql import functions as F

# 1. Preparar el dataset con rangos y uniones
df_biv_extendido = spark.table("df_fact_silver").join(
    spark.table("dim_clientes"),
    "ID_Cliente",
    "inner"
).withColumn(
    "Rango_Edad",
    F.when(F.col("Edad") < 30, "18-29")
     .when(F.col("Edad") < 45, "30-44")
     .when(F.col("Edad") < 60, "45-59")
     .otherwise("60+")
)

# 2. Función para análisis de 2 variables + Churn
def analizar_cruce_bivariado(df, var1, var2):
    print(f"\n--- Análisis de Abandono: {var1} + {var2} ---")
    df.groupBy(var1, var2).agg(
        F.countDistinct("ID_Cliente").alias("Total_Clientes"),
        F.sum("Es_Abandono").alias("Abandonos"),
        F.round((F.sum("Es_Abandono") / F.countDistinct("ID_Cliente") * 100), 2).alias("Tasa_Churn_%")
    ).orderBy(F.desc("Tasa_Churn_%")).show(20)

# 3. Ejecución de los cruces (Originales + Adicionales)
# Cruces iniciales
analizar_cruce_bivariado(df_biv_extendido, "Segmento", "Canal_Venta")
analizar_cruce_bivariado(df_biv_extendido, "Producto_Principal", "Departamento")

# Cruces adicionales solicitados
print("\n>>> Cruces Adicionales de Profundización <<<")
analizar_cruce_bivariado(df_biv_extendido, "Segmento", "Rango_Edad")
analizar_cruce_bivariado(df_biv_extendido, "Producto_Principal", "Canal_Venta")
analizar_cruce_bivariado(df_biv_extendido, "Departamento", "Canal_Venta")


--- Análisis de Abandono: Segmento + Canal_Venta ---
+------------+-------------------+--------------+---------+------------+
|    Segmento|        Canal_Venta|Total_Clientes|Abandonos|Tasa_Churn_%|
+------------+-------------------+--------------+---------+------------+
|     Digital|        Call Center|        210383|    79905|       37.98|
|     Digital|         Página Web|        256538|    89336|       34.82|
|     Digital|  Alianza Comercial|         69732|    24232|       34.75|
|     Digital|      Banca Digital|        373872|   128505|       34.37|
|     Clásico|        Call Center|        420396|   136581|       32.49|
|     Digital|            Agencia|        139828|    42908|       30.69|
|     Digital|Ejecutivo Comercial|        117061|    32969|       28.16|
|Preferencial|        Call Center|        141723|    38454|       27.13|
|     Clásico|  Alianza Comercial|        279937|    67310|       24.04|
|     Clásico|         Página Web|        419837|    98281|       23.4

In [ ]:
!zip -r /content/archivos_parquet.zip /content/churn_banca/silver


updating: content/churn_banca/silver/dim_clientes/ (stored 0%)
updating: content/churn_banca/silver/dim_clientes/.part-00250-918b40eb-b4b1-4d71-9694-ebabebb3d6ad-c000.snappy.parquet.crc (deflated 10%)
updating: content/churn_banca/silver/dim_clientes/part-00079-918b40eb-b4b1-4d71-9694-ebabebb3d6ad-c000.snappy.parquet (deflated 56%)
updating: content/churn_banca/silver/dim_clientes/.part-00195-918b40eb-b4b1-4d71-9694-ebabebb3d6ad-c000.snappy.parquet.crc (deflated 10%)
updating: content/churn_banca/silver/dim_clientes/.part-00395-918b40eb-b4b1-4d71-9694-ebabebb3d6ad-c000.snappy.parquet.crc (deflated 11%)
updating: content/churn_banca/silver/dim_clientes/.part-00165-918b40eb-b4b1-4d71-9694-ebabebb3d6ad-c000.snappy.parquet.crc (deflated 12%)
updating: content/churn_banca/silver/dim_clientes/.part-00088-918b40eb-b4b1-4d71-9694-ebabebb3d6ad-c000.snappy.parquet.crc (deflated 12%)
updating: content/churn_banca/silver/dim_clientes/.part-00150-918b40eb-b4b1-4d71-9694-ebabebb3d6ad-c000.snappy.par

In [ ]:
from google.colab import files
files.download('/content/archivos_parquet.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!du -sh /content/churn_banca/silver/fact_silver/


467M	/content/churn_banca/silver/fact_silver/
